## regrid ST4 data to PRISM grid using XESMF bilinear method, write to zarr

### I ran several tests of this method vs the "budget" method of wgrib2 and the native ST4 grid. The native grid does identify some events that the regridded data do not, but it tended to be single points on single days, which are probably ok to limit anyway. So the regridding isn't perfect (that's not really possible) but seems sufficient for our purposes here.

### And unfortunately the 'conservative' regridding method in xesmf doesn't work for curvilinear grids like HRAP, so that option doesn't work - so will use bilinear for all datasets

In [2]:
import xarray as xr
import pandas as pd
import xesmf as xe
import numpy as np
from pathlib import Path

In [3]:
### open file to mask for all-nan points
nanmask = xr.open_dataset("stage4_nanmask_prismgrid.nc")['tp']

In [4]:
### open a prism file to get the crs information
prismfile = xr.open_dataset("/data/rschumac/prism/daily/PRISM_ppt_stable_4kmD2_20240622_bil.nc")


In [5]:
duration = "24" ## in hours (24, 06, etc.)

for year in range(2002,2025):
#for year in range(2024,2025):

    print("working on "+str(year))

    if year==2020:
        data1 = xr.open_dataset("/home/rschumac/precip_data/stage4/ST4_annual_"+duration+"h_"+str(year)+".grb2", 
                                engine='cfgrib', indexpath='',
                                filter_by_keys={'dataType':'fc'})
        data2 = xr.open_dataset("/home/rschumac/precip_data/stage4/ST4_annual_"+duration+"h_"+str(year)+".grb2",
                                engine='cfgrib', indexpath='',
                                filter_by_keys={'dataType':'an'})
        data2['longitude'] = data1['longitude'] ### make sure these match exactly
        data2['latitude'] = data1['latitude'] 
        data = xr.concat([data1,data2],dim='time')

    else:
        data = xr.open_dataset("/home/rschumac/precip_data/stage4/ST4_annual_"+duration+"h_"+str(year)+".grb2", 
                           engine='cfgrib', indexpath='')

    ### reshape the data to set the valid time as the time dimension
    data_reshape = xr.DataArray(data = data.tp.values,
                        dims=["time", "x", "y"],
                        coords=dict(
                        lon=(["x", "y"], data.longitude.values),
                        lat=(["x", "y"], data.latitude.values),
                        time=pd.to_datetime(pd.to_datetime(data.valid_time.values))))
    data_reshape = data_reshape.to_dataset(name='tp')

    ### build regridder
    ds_out = xr.Dataset(
        {
            "lat": (["lat"], np.arange(24.08333, 49.91667, 0.04166667)),
            "lon": (["lon"], np.arange(-125, -66.499, 0.04166667)),
        }
    )
    
    weights_file = Path('bilinear_881x1121_621x1405.nc')
    if weights_file.is_file():
        regridder = xe.Regridder(data_reshape, ds_out, "bilinear", weights='bilinear_881x1121_621x1405.nc')
    else:
        regridder = xe.Regridder(data_reshape, ds_out, "bilinear")
        regridder.to_netcdf()
    
    ### do the regridding
    data_out = regridder(data_reshape)

    ### apply the mask to set all the zero points to nan
    data_out = data_out.where(nanmask==1)

    ### set a few attributes
    data_out.attrs['description'] = "NCEP Stage IV multi-sensor precipitation analysis, "+str(duration)+"-h accumulations, regridded to the 4-km PRISM grid using xESMF's bilinear method"
    data_out.attrs['source'] = "Original data source: Du, J. 2011. NCEP/EMC 4KM Gridded Data (GRIB) Stage IV Data. Version 1.0. UCAR/NCAR - Earth Observing Laboratory. https://doi.org/10.5065/D6PG1QDD"
    data_out.tp.attrs['long_name'] = "accumulated precipitation"
    data_out.tp.attrs['units'] = "mm"
    data_out.tp.attrs['time_desc'] = "Values represent the precipitation in the "+str(duration)+" hours ending at the time given in the dataset"
    data_out.time.attrs['long_name'] = "valid time (UTC)"
    data_out.time.attrs['description'] = "Time represents the end of a "+str(duration)+"-hour accumulation period"
    data_out.lon.attrs['long_name'] = "longitude"
    data_out.lon.attrs['standard_name'] = "longitude"
    data_out.lon.attrs['units'] = "degrees_east"
    data_out.lat.attrs['long_name'] = "latitude"
    data_out.lat.attrs['standard_name'] = "latitude"
    data_out.lat.attrs['units'] = "degrees_north"

    data_out['crs'] = prismfile['crs'] ### add crs and its attributes as well

    ### set encoding for compression, write netcdf file
    #comp = dict(zlib=True, complevel=7)
    #encoding = {var: comp for var in data_out.data_vars}
    #data_out.to_netcdf("stage4_annual_"+duration+"h_"+str(year)+"_prismgrid.nc", encoding=encoding)

    ### if first year, create zarr store, otherwise append to existing store
    if year==2002:
        data_out.to_zarr("/data/rschumac/gridded_precip_zarr/stage4_"+duration+"h_prismgrid.zarr",
                        mode='w')
    else:
        data_out.to_zarr("/data/rschumac/gridded_precip_zarr/stage4_"+duration+"h_prismgrid.zarr",
                append_dim='time')

print("done!")

working on 2002
working on 2003
working on 2004
working on 2005
working on 2006
working on 2007
working on 2008
working on 2009
working on 2010
working on 2011
working on 2012
working on 2013
working on 2014
working on 2015
working on 2016
working on 2017
working on 2018
working on 2019
working on 2020
working on 2021
working on 2022
working on 2023
working on 2024


[WARNING] yaksa: 10 leaked handle pool objects


In [7]:
data_out

<xarray.Dataset>
Dimensions:  (time: 366, lat: 621, lon: 1405)
Coordinates:
  * time     (time) datetime64[ns] 2024-01-01T12:00:00 ... 2024-12-31T12:00:00
  * lat      (lat) float64 24.08 24.12 24.17 24.21 ... 49.79 49.83 49.87 49.92
  * lon      (lon) float64 -125.0 -125.0 -124.9 -124.9 ... -66.58 -66.54 -66.5
Data variables:
    tp       (time, lat, lon) float32 nan nan nan nan nan ... nan nan nan nan
    crs      |S1 ...
Attributes:
    regrid_method:  bilinear
    description:    NCEP Stage IV multi-sensor precipitation analysis, 24-h a...
    source:         Original data source: Du, J. 2011. NCEP/EMC 4KM Gridded D...

[WARNING] yaksa: 10 leaked handle pool objects


In [ ]:
#### native ST4 file metadata for reference

#data = data.metpy.assign_crs(grid_mapping_name="polar_stereographic",
#                   straight_vertical_longitude_from_pole=-105.0,
#                   false_easting = 0.,
#                   standard_parallel = 60.,
#                   false_northing = 0.0,
#                   latitude_of_projection_origin = 90.0,
#                   earth_radius = 6371200.0)

data